In [41]:
import pandas as pd
import unicodedata
import glob
import os


### Sumário de padrões para tratamento de dados:

- Data: DD/MM/YYYY

- Data hora: DD/MM/YYYY HH:MM:SS

- Binário: 0 ou 1

- Strings e colunas: Maiusculas e sem acento: 

- Espaços: Manter apenas um espaço no intervalo entre palavras

- Sexo: M e F

- Decimais: Usar pontos

- Delimitador: Vírgula

- Armazenamento: Todos os arquivos em .csv


### Funções para padronização:

In [42]:
# Formato em 2016-04-29T18:38:08Z
def padronizar_data_hora(df, coluna):

  df[coluna] = pd.to_datetime(df[coluna])
  
  df[coluna] = df[coluna].dt.strftime('%d/%m/%Y %H:%M:%S')
  
  return df


In [43]:
#Formato em MM/DD/AA
def padronizar_data(df, coluna):

  df[coluna] = pd.to_datetime(df[coluna], format='%m/%d/%Y')
  
  df[coluna] = df[coluna].dt.strftime('%d/%m/%Y')
  
  return df

In [44]:
def padronizar_data2(df, coluna):
  
    df[coluna] = pd.to_datetime(df[coluna], format='%Y-%m-%d')
    
    df[coluna] = df[coluna].dt.strftime('%d/%m/%Y')
    
    return df

In [45]:
def padronizar_colunas(df):

    df.columns = df.columns.str.upper()
    
    return df

In [46]:
def converter_para_binario(df, coluna):
    mapeamento = {'Yes': 1, 'No': 0}
    df[coluna].replace(mapeamento, inplace=True)
    return df

In [47]:
def remover_acentos(df):
    for coluna in df.columns:
        if df[coluna].dtype == 'object':
            df[coluna] = df[coluna].astype(str).str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    return df


In [48]:
def padronizar_maiusculo(df):
    for coluna in df.columns:
        if df[coluna].dtype == 'object':
            df[coluna] = df[coluna].astype(str).str.upper()
    return df

In [49]:

def padronizar_decimal_para_ponto(df):

    colunas_string = df.select_dtypes(include=['object']).columns
    
    for coluna in colunas_string:

        coluna_limpa = df[coluna].astype(str).str.replace(',', '.', regex=False)
        

        coluna_convertida = pd.to_numeric(coluna_limpa, errors='coerce')
        
        limiar_sucesso = 0.8
        
        if coluna_convertida.count() / len(coluna_convertida) > limiar_sucesso:
            df[coluna] = coluna_convertida
            
    return df

### Tratamento de dados

In [50]:
df_med = pd.read_csv("dados/raw/medical_appointments.csv")

In [51]:

df_clima = pd.read_csv('dados/raw/meteorologia2016.csv', sep=';')



In [52]:
df_med = padronizar_data_hora(df_med, 'ScheduledDay')
df_med = padronizar_data_hora(df_med, 'AppointmentDay')
df_med = padronizar_colunas(df_med)
df_med = converter_para_binario(df_med, 'NO-SHOW')
df_med = remover_acentos(df_med)
df_med = padronizar_maiusculo(df_med)

C:\Users\ivanm\AppData\Local\Temp\ipykernel_19084\953233055.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[coluna].replace(mapeamento, inplace=True)
C:\Users\ivanm\AppData\Local\Temp\ipykernel_19084\953233055.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[coluna].replace(mapeamento, 

In [53]:
df_med = df_med[df_med['AGE'] >= 0]

In [54]:

df_clima.columns = [
    "DATA", "HORA_UTC", "PRECIPITACAO_MM", "PRESSAO_ESTACAO_MB", "PRESSAO_MAX_MB",
    "PRESSAO_MIN_MB", "RADIACAO_KJ_M2", "TEMP_AR_C", "TEMP_ORVALHO_C", "TEMP_MAX_C",
    "TEMP_MIN_C", "TEMP_ORVALHO_MAX_C", "TEMP_ORVALHO_MIN_C", "UMIDADE_MAX",
    "UMIDADE_MIN", "UMIDADE_RELATIVA", "VENTO_DIRECAO_GRAUS", "VENTO_RAJADA_MAX_MS",
    "VENTO_VELOCIDADE_MS", "DESCARTAR"
]
df_clima = df_clima.drop(columns=["DESCARTAR"])



In [55]:
df_clima = padronizar_data2(df_clima, 'DATA')

In [56]:
df_clima

,DATA,HORA_UTC,PRECIPITACAO_MM,PRESSAO_ESTACAO_MB,PRESSAO_MAX_MB,PRESSAO_MIN_MB,RADIACAO_KJ_M2,TEMP_AR_C,TEMP_ORVALHO_C,TEMP_MAX_C,TEMP_MIN_C,TEMP_ORVALHO_MAX_C,TEMP_ORVALHO_MIN_C,UMIDADE_MAX,UMIDADE_MIN,UMIDADE_RELATIVA,VENTO_DIRECAO_GRAUS,VENTO_RAJADA_MAX_MS,VENTO_VELOCIDADE_MS
0,01/01/2016,00:00,0,"924,1","924,1","923,6",-9999,"23,8","19,1","25,1","23,8","19,2","18,8",75,69,75,322,7,"3,4"
1,01/01/2016,01:00,0,"924,4","924,5","924,1",-9999,23,"18,7","23,8",23,"19,2","18,7",77,75,77,325,7,"3,7"
2,01/01/2016,02:00,0,"924,1","924,6","924,1",-9999,"22,5","18,4","23,1","22,4","18,8","18,4",78,76,77,334,"7,8","3,3"
3,01/01/2016,03:00,0,"923,9","924,1","923,8",-9999,"22,2","18,2","22,5",22,"18,4","18,1",79,77,78,306,"7,3","2,2"
4,01/01/2016,04:00,0,"923,3",924,"923,3",-9999,"21,7",18,"22,2","21,7","18,2",18,79,78,79,322,"7,7","4,2"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8779,31/12/2016,19:00,0,"923,3","924,2",923,"442,2","26,3","16,7","26,3","25,3",18,"16,5",63,56,56,307,"6,6","1,7"
8780,31/12/2016,20:00,0,"923,4","923,5","923,1","544,8","27,7","16,1","27,7","26,3","16,5","14,6",55,46,49,353,"5,1","1,6"
8781,31/12/2016,21:00,0,"923,6",924,"923,4","195,4","27,1","16,4","27,7",27,"16,5","15,9",53,49,52,43,"4,5",",7"
8782,31/12/2016,22:00,0,"924,1","924,3","923,6","29,3","26,6","16,8","27,2","26,6","16,8","16,2",55,51,55,331,"3,4","1,1"


In [57]:
df_clima = padronizar_decimal_para_ponto(df_clima)

In [58]:
df_clima.head()

,DATA,HORA_UTC,PRECIPITACAO_MM,PRESSAO_ESTACAO_MB,PRESSAO_MAX_MB,PRESSAO_MIN_MB,RADIACAO_KJ_M2,TEMP_AR_C,TEMP_ORVALHO_C,TEMP_MAX_C,TEMP_MIN_C,TEMP_ORVALHO_MAX_C,TEMP_ORVALHO_MIN_C,UMIDADE_MAX,UMIDADE_MIN,UMIDADE_RELATIVA,VENTO_DIRECAO_GRAUS,VENTO_RAJADA_MAX_MS,VENTO_VELOCIDADE_MS
0,01/01/2016,00:00,0.0,924.1,924.1,923.6,-9999.0,23.8,19.1,25.1,23.8,19.2,18.8,75,69,75,322,7.0,3.4
1,01/01/2016,01:00,0.0,924.4,924.5,924.1,-9999.0,23.0,18.7,23.8,23.0,19.2,18.7,77,75,77,325,7.0,3.7
2,01/01/2016,02:00,0.0,924.1,924.6,924.1,-9999.0,22.5,18.4,23.1,22.4,18.8,18.4,78,76,77,334,7.8,3.3
3,01/01/2016,03:00,0.0,923.9,924.1,923.8,-9999.0,22.2,18.2,22.5,22.0,18.4,18.1,79,77,78,306,7.3,2.2
4,01/01/2016,04:00,0.0,923.3,924.0,923.3,-9999.0,21.7,18.0,22.2,21.7,18.2,18.0,79,78,79,322,7.7,4.2


In [59]:
df_med.to_csv('dados/trusted/medical_appointment_no_show.csv', index=False)

In [60]:
df_clima.to_csv('dados/trusted/clima.csv', index=False)    